# 08 - E7b conservative test-time adaptation

E7b avoids the BatchNorm batch-statistics collapse seen in E7 v1. It adapts only
the **LayerNorm** affine parameters of the VQSA self-attention blocks (batch
agnostic) while keeping every BatchNorm on its source running statistics, and it
adds the DeMemte **latent-memory preservation** regularizer (preserve `z`, `zq`,
codebook soft-assignments against a frozen source teacher) — pattern completion,
not pure entropy collapse.

Methods (TTA reset at the start of each clean/corruption/severity condition):

- `source` — raw E6 winner, no adaptation.
- `bn_stats_no_update` — literature "BN Stats" baseline (per-batch BN stats, **no**
  gradient step). Reported next to `source`; expected to collapse here.
- `tent_ln` — TENT entropy minimization over LayerNorm only.
- `eata_ln` — EATA-lite reliability/diversity filter over LayerNorm.
- `eata_ln_srcfilter` — EATA whose filter reads the frozen source teacher logits.
- `tent_ln_memreg` — `tent_ln` + latent-memory preservation.
- `eata_ln_srcfilter_memreg` — teacher-filtered EATA + latent-memory preservation.


In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'src' / 'dememte').exists():
    ROOT = ROOT.parent
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
print('repo root:', ROOT)

repo root: /home/nakato/projects/Dememte


In [2]:
from dataclasses import asdict
import math

import numpy as np
import pandas as pd
import torch

from dememte.config import E6_SPECS, e6_config
from dememte.data import build_loaders, seed_everything
from dememte.evaluation import evaluate_dememte_suite, evaluate_dememte_tta_suite, signal_curve_rows
from dememte.io import ensure_dir, load_checkpoint, write_csv, write_json
from dememte.models import make_dememte_e6
from dememte.tta import (
    EATALiteAdapter,
    MemoryTentAdapter,
    NoUpdateAdapter,
    SourceFilterEATAAdapter,
    TentAdapter,
    collect_tta_bn_params,
    collect_tta_ln_params,
    configure_tta_layernorm,
    configure_tta_model,
    make_tta_optimizer,
)

BASE_VARIANT = 'e6_ema_kmeans_restart'
METHODS = [
    'source',
    'bn_stats_no_update',
    'tent_ln',
    'eata_ln',
    'eata_ln_srcfilter',
    'tent_ln_memreg',
    'eata_ln_srcfilter_memreg',
]

# TTA hyperparameters (shared with E7 v1; sweep later, not in this first cut).
TTA_LR = 2.5e-4
TTA_MOMENTUM = 0.9
EATA_D_MARGIN = 0.05
# Latent-memory preservation weights: (w_z, w_zq, w_assign).
MEM_WEIGHTS = (1.0, 1.0, 1.0)

OUT = ensure_dir(ROOT / 'notebooks' / '08_e7b_tta' / 'out')
E6_OUT = ROOT / 'notebooks' / '06_e6_zq_alignment' / 'out'
CKPT = E6_OUT / BASE_VARIANT / 'best.pt'

cfg = e6_config(BASE_VARIANT)
for candidate in [ROOT / 'experiments' / 'data', ROOT / 'data', Path(cfg.data_dir).expanduser()]:
    candidate = candidate.resolve()
    if (candidate / 'flowers-102').exists() or candidate.name == 'flowers-102':
        cfg.data_dir = str(candidate)
        break

device = 'cuda' if torch.cuda.is_available() else 'cpu'
seed_everything(cfg.seed)
print('device:', device)
print('checkpoint:', CKPT)
print('eata entropy margin:', 0.4 * math.log(cfg.num_classes))

device: cuda
checkpoint: /home/nakato/projects/Dememte/notebooks/06_e6_zq_alignment/out/e6_ema_kmeans_restart/best.pt
eata entropy margin: 1.8499891253137084


## Data

In [3]:
tr_loader, va_loader, te_loader, meta = build_loaders(
    data_dir=cfg.data_dir,
    batch_size=cfg.batch_size,
    num_workers=cfg.num_workers,
    val_ratio=cfg.val_ratio,
    split_seed=cfg.split_seed,
    protocol=cfg.benchmark_protocol,
)
print(meta)

{'protocol': 'historical_trainval_resplit', 'split_seed': 42, 'train_size': 1632, 'val_size': 408, 'test_size': 6149}


## Evaluation helpers

In [4]:
def write_markdown_table(rows, path):
    path = Path(path)
    ensure_dir(path.parent)
    if not rows:
        path.write_text('', encoding='utf-8')
        return
    df = pd.DataFrame(rows)
    path.write_text(df.to_markdown(index=False), encoding='utf-8')


def load_base_model():
    model = make_dememte_e6(cfg, device=device)
    load_checkpoint(model, CKPT, device=device, strict=True)
    return model


def make_adapter(method):
    # Reported "BN Stats" baseline: per-batch BN stats, no gradient step.
    if method == 'bn_stats_no_update':
        model = configure_tta_model(load_base_model())
        params, _ = collect_tta_bn_params(model)
        opt = make_tta_optimizer(params, lr=TTA_LR, momentum=TTA_MOMENTUM)
        return NoUpdateAdapter(model, opt)

    # E7b adaptation surface: LayerNorm only, BN stays on source running stats.
    model = configure_tta_layernorm(load_base_model())
    params, names = collect_tta_ln_params(model)
    if not params:
        raise RuntimeError('No LayerNorm affine params found (needs vqsa_use_self_attention=True)')
    opt = make_tta_optimizer(params, lr=TTA_LR, momentum=TTA_MOMENTUM)

    if method == 'tent_ln':
        return TentAdapter(model, opt, steps=1, episodic=False)
    if method == 'eata_ln':
        return EATALiteAdapter(
            model, opt, num_classes=cfg.num_classes, steps=1, episodic=False, d_margin=EATA_D_MARGIN,
        )
    if method == 'eata_ln_srcfilter':
        return SourceFilterEATAAdapter(
            model, opt, num_classes=cfg.num_classes, source_model=load_base_model(),
            steps=1, episodic=False, d_margin=EATA_D_MARGIN,
        )
    if method == 'tent_ln_memreg':
        return MemoryTentAdapter(
            model, opt, source_model=load_base_model(), steps=1, episodic=False,
            w_z=MEM_WEIGHTS[0], w_zq=MEM_WEIGHTS[1], w_assign=MEM_WEIGHTS[2],
        )
    if method == 'eata_ln_srcfilter_memreg':
        return SourceFilterEATAAdapter(
            model, opt, num_classes=cfg.num_classes, source_model=load_base_model(),
            steps=1, episodic=False, d_margin=EATA_D_MARGIN, memory_weights=MEM_WEIGHTS,
        )
    raise ValueError(method)


def summarize_metrics(method, metrics):
    summary = {k: v for k, v in metrics.items() if isinstance(v, (int, float, bool, str, np.floating))}
    summary.update({
        'variant': method,
        'label': method,
        'base_variant': BASE_VARIANT,
        'base_checkpoint': str(CKPT),
        'protocol': meta['protocol'],
        'split_seed': meta['split_seed'],
        'quantizer_type': cfg.quantizer_type,
    })
    return summary

## Run E7b

In [5]:
if not CKPT.exists():
    raise FileNotFoundError(f'Missing E6 winner checkpoint: {CKPT}')

all_summaries = []
all_curves = []

for method in METHODS:
    print(f'=== {method} ===')
    if method == 'source':
        model = load_base_model()
        metrics = evaluate_dememte_suite(model, te_loader, device=device)
    else:
        metrics = evaluate_dememte_tta_suite(
            lambda method=method: make_adapter(method),
            te_loader,
            device=device,
            tta_method=method,
            tta_base_variant=BASE_VARIANT,
        )

    clean_record = metrics.pop('clean_record')
    corrupt_records = metrics.pop('corruption_records')
    curve_rows = signal_curve_rows(method, method, clean_record, corrupt_records)
    summary = summarize_metrics(method, metrics)
    all_summaries.append(summary)
    all_curves.extend(curve_rows)

    method_dir = ensure_dir(OUT / method)
    write_json(summary, method_dir / 'metrics.json')
    write_csv(curve_rows, method_dir / 'signal_curves.csv')
    report_keys = [
        'clean_acc', 'corrupt_acc_avg', 'ece_corrupt_avg', 'nll_corrupt_avg',
        'hard_usage_corrupt_avg', 'dead_code_fraction_corrupt_avg',
    ]
    print({k: round(float(summary[k]), 4) for k in report_keys if k in summary})

write_csv(all_summaries, OUT / 'e7b_results.csv')
write_csv(all_curves, OUT / 'e7b_curves.csv')

ranked = sorted(all_summaries, key=lambda r: r.get('corrupt_acc_avg', 0.0), reverse=True)
write_markdown_table(ranked, OUT / 'e7b_summary.md')
pd.DataFrame(ranked)

=== source ===
{'clean_acc': 0.7523, 'corrupt_acc_avg': 0.503, 'ece_corrupt_avg': 0.0903, 'nll_corrupt_avg': 2.0224, 'hard_usage_corrupt_avg': 0.4792, 'dead_code_fraction_corrupt_avg': 0.5208}
=== bn_stats_no_update ===
{'clean_acc': 0.0317, 'corrupt_acc_avg': 0.0284, 'ece_corrupt_avg': 0.4896, 'nll_corrupt_avg': 8.7399, 'hard_usage_corrupt_avg': 0.4583, 'dead_code_fraction_corrupt_avg': 0.5417}
=== tent_ln ===
{'clean_acc': 0.752, 'corrupt_acc_avg': 0.5021, 'ece_corrupt_avg': 0.0948, 'nll_corrupt_avg': 2.0328, 'hard_usage_corrupt_avg': 0.4792, 'dead_code_fraction_corrupt_avg': 0.5208}
=== eata_ln ===
{'clean_acc': 0.7517, 'corrupt_acc_avg': 0.5037, 'ece_corrupt_avg': 0.0912, 'nll_corrupt_avg': 2.0211, 'hard_usage_corrupt_avg': 0.4792, 'dead_code_fraction_corrupt_avg': 0.5208}
=== eata_ln_srcfilter ===
{'clean_acc': 0.752, 'corrupt_acc_avg': 0.5036, 'ece_corrupt_avg': 0.0913, 'nll_corrupt_avg': 2.0213, 'hard_usage_corrupt_avg': 0.4792, 'dead_code_fraction_corrupt_avg': 0.5208}
=== tent

,clean_acc,corrupt_acc_avg,corrupt_acc_gaussian_noise,corrupt_acc_pixel_mask,corrupt_acc_cutout,corrupt_acc_blur,ece_clean,ece_corrupt_avg,nll_clean,nll_corrupt_avg,...,dead_code_fraction_corrupt_avg,attention_entropy_clean,attention_entropy_corrupt_avg,variant,label,base_variant,base_checkpoint,protocol,split_seed,quantizer_type
0,0.751667,0.503686,0.353445,0.350030,0.639508,0.671762,0.059129,0.091199,0.977523,2.021128,...,0.520752,0.778140,0.805807,eata_ln,eata_ln,e6_ema_kmeans_restart,/home/nakato/projects/Dememte/notebooks/06_e6_...,historical_trainval_resplit,42,ema_vq
1,0.751992,0.503578,0.353445,0.349921,0.639345,0.671600,0.058809,0.091269,0.977546,2.021289,...,0.520752,0.778140,0.805794,eata_ln_srcfilter,eata_ln_srcfilter,e6_ema_kmeans_restart,/home/nakato/projects/Dememte/notebooks/06_e6_...,historical_trainval_resplit,42,ema_vq
2,0.751992,0.503578,0.353445,0.349921,0.639345,0.671600,0.058809,0.091269,0.977546,2.021289,...,0.520752,0.778140,0.805794,eata_ln_srcfilter_memreg,eata_ln_srcfilter_memreg,e6_ema_kmeans_restart,/home/nakato/projects/Dememte/notebooks/06_e6_...,historical_trainval_resplit,42,ema_vq
3,0.752317,0.502954,0.353445,0.348675,0.638911,0.670787,0.058221,0.090268,0.976969,2.022439,...,0.520752,0.778217,0.805808,source,source,e6_ema_kmeans_restart,/home/nakato/projects/Dememte/notebooks/06_e6_...,historical_trainval_resplit,42,ema_vq
4,0.751992,0.502074,0.352415,0.345530,0.639345,0.671003,0.058575,0.094805,0.977741,2.032763,...,0.520752,0.778154,0.805613,tent_ln,tent_ln,e6_ema_kmeans_restart,/home/nakato/projects/Dememte/notebooks/06_e6_...,historical_trainval_resplit,42,ema_vq
5,0.751992,0.502074,0.352415,0.345530,0.639345,0.671003,0.058575,0.094805,0.977741,2.032763,...,0.520752,0.778154,0.805613,tent_ln_memreg,tent_ln_memreg,e6_ema_kmeans_restart,/home/nakato/projects/Dememte/notebooks/06_e6_...,historical_trainval_resplit,42,ema_vq
6,0.031712,0.028392,0.026725,0.022768,0.032200,0.031875,0.491803,0.489564,8.663169,8.739865,...,0.541748,0.734276,0.732183,bn_stats_no_update,bn_stats_no_update,e6_ema_kmeans_restart,/home/nakato/projects/Dememte/notebooks/06_e6_...,historical_trainval_resplit,42,ema_vq


## Read

**Sanity gate (technical, not scientific).** `tent_ln` must not collapse: it
should land near or above `source` (E6 winner: clean ≈ 0.7523, corrupt ≈ 0.5030).
If it does, the failure in E7 v1 was the BatchNorm batch-statistics surface, not
TENT/EATA per se. `bn_stats_no_update` is expected to collapse and is reported as
a baseline (it is not a gate that invalidates the experiment).

**Memory preservation (Q5).** Compare `hard_usage_corrupt_avg` and
`dead_code_fraction_corrupt_avg` of the `*_memreg` variants against `source`: the
regularizer should keep the codebook usage close to source while adapting, i.e.
the latent memory is preserved rather than destroyed.